In [1]:
# I am installing the packages needed for this regression notebook.
# JupyterLite sometimes resets packages, so I am starting clean.

import piplite

await piplite.install(["pandas", "openpyxl", "statsmodels"])

In [2]:
# I am loading the main Python libraries for this analysis.
# pandas handles the Excel data, numpy helps with log transformations,
# statsmodels runs the regression, and os checks the uploaded files.

import pandas as pd
import numpy as np
import statsmodels.api as sm
import os

In [3]:
# I am checking the exact Excel file name uploaded in JupyterLite.
# This avoids file loading errors caused by small spelling differences.

excel_files = [file for file in os.listdir() if file.endswith(".xlsx")]

excel_files

['Main_Datasheet_Rem.xlsx']

In [4]:
# I am loading the updated Excel workbook with the remittance variables.
# The file name below is taken directly from the uploaded files list.

file_name = "Main_Datasheet_Rem.xlsx"

df = pd.read_excel(file_name, sheet_name="Analysis_Data")

print(df.shape)
df.head()

(300, 29)


,Constituency,Districts,Upazilas,Upazila_Count,Area_sqkm,Population,Population_Density,Household_Size,Literacy_Rate,Crime_Units,...,Runner_Up_Votes,Runner_Up_Vote_Share,Margin_Votes,Margin_Percentage,Competitiveness,Metro,Use_For_Analysis,Remittance_HH_Total,CHECK,Remittance_HH_Urban
0,Bagerhat-1,Bagerhat,"Fakirhat, Mollahat, Chitalmari",3,813.53,456659,561.330252,4.072584,79.895851,Khulna Range,...,114323,49.309036,3204,1.381928,Highly Competitive,0,1,13393,9102,4291
1,Bagerhat-2,Bagerhat,"Bagerhat Sadar, Kachua",2,631.00,694132,1100.050713,4.053564,78.172080,Khulna Range,...,66409,36.068717,51300,27.862566,Safe Seat,0,1,13393,9102,4291
2,Bagerhat-3,Bagerhat,"Rampal, Mongla",2,1753.34,334508,190.783305,3.792103,81.073165,Khulna Range,...,83550,44.868456,19111,10.263089,Safe Seat,0,1,13393,9102,4291
3,Bagerhat-4,Bagerhat,"Morelganj, Sharankhola",2,624.80,425769,681.448464,3.801240,82.420112,Khulna Range,...,98326,45.862505,17741,8.274990,Safe Seat,0,1,13393,9102,4291
4,Bandarban-1,Bandarban,"Bandarban Sadar, Thanchi, Lama, Naikhongchhari...",7,4304.43,481093,111.766947,4.417808,63.630036,Chittagong Range,...,26162,15.608202,115293,68.783596,Safe Seat,0,1,4106,2100,2006


In [5]:
# I am checking all column names before modelling.
# This helps confirm that the remittance columns were added correctly.

df.columns.tolist()

['Constituency',
 'Districts',
 'Upazilas',
 'Upazila_Count',
 'Area_sqkm',
 'Population',
 'Population_Density',
 'Household_Size',
 'Literacy_Rate',
 'Crime_Units',
 'Total_Crime_per_100k',
 'Violent_Crime_per_100k',
 'Election_Status',
 'Winner',
 'Winning_Party',
 'Winning_Party_Code',
 'Winner_Votes',
 'Winner_Vote_Share',
 'Runner_Up',
 'Runner_Up_Votes',
 'Runner_Up_Vote_Share',
 'Margin_Votes',
 'Margin_Percentage',
 'Competitiveness',
 'Metro',
 'Use_For_Analysis',
 'Remittance_HH_Total',
 'CHECK',
 'Remittance_HH_Urban']

In [6]:
# One remittance column was accidentally imported with the header CHECK.
# Based on the column order, this is the rural remittance household column.
# I am renaming it so the model has clear variable names.

if "CHECK" in df.columns:
    df = df.rename(columns={"CHECK": "Remittance_HH_Rural"})

df.columns.tolist()

['Constituency',
 'Districts',
 'Upazilas',
 'Upazila_Count',
 'Area_sqkm',
 'Population',
 'Population_Density',
 'Household_Size',
 'Literacy_Rate',
 'Crime_Units',
 'Total_Crime_per_100k',
 'Violent_Crime_per_100k',
 'Election_Status',
 'Winner',
 'Winning_Party',
 'Winning_Party_Code',
 'Winner_Votes',
 'Winner_Vote_Share',
 'Runner_Up',
 'Runner_Up_Votes',
 'Runner_Up_Vote_Share',
 'Margin_Votes',
 'Margin_Percentage',
 'Competitiveness',
 'Metro',
 'Use_For_Analysis',
 'Remittance_HH_Total',
 'Remittance_HH_Rural',
 'Remittance_HH_Urban']

In [7]:
# I am checking how many constituencies are marked as valid for analysis.
# The expected final regression sample is 297 constituencies.

df["Use_For_Analysis"].value_counts(dropna=False)

Use_For_Analysis
1    297
0      3
Name: count, dtype: int64

In [8]:
# I am preparing the first regression dataset.
# This model uses the original predictors from the first OLS analysis.

model_1_vars = [
    "Winner_Vote_Share",
    "Population_Density",
    "Household_Size",
    "Literacy_Rate",
    "Total_Crime_per_100k",
    "Violent_Crime_per_100k",
    "Metro"
]

reg_df_1 = df.loc[df["Use_For_Analysis"] == 1, model_1_vars].copy()

for col in model_1_vars:
    reg_df_1[col] = pd.to_numeric(reg_df_1[col], errors="coerce")

print(reg_df_1.shape)
reg_df_1.isna().sum()

(297, 7)


Winner_Vote_Share         0
Population_Density        0
Household_Size            0
Literacy_Rate             0
Total_Crime_per_100k      0
Violent_Crime_per_100k    0
Metro                     0
dtype: int64

In [9]:
# Model 1 tests whether population density, household size, literacy,
# crime proxies, and metro status are associated with winner vote share.
# I am using robust standard errors because constituency-level data may have uneven error patterns.

y1 = reg_df_1["Winner_Vote_Share"]

X1 = reg_df_1[
    [
        "Population_Density",
        "Household_Size",
        "Literacy_Rate",
        "Total_Crime_per_100k",
        "Violent_Crime_per_100k",
        "Metro"
    ]
]

X1 = sm.add_constant(X1)

model_1_robust = sm.OLS(y1, X1).fit(cov_type="HC3")

print(model_1_robust.summary())

                            OLS Regression Results                            
Dep. Variable:      Winner_Vote_Share   R-squared:                       0.101
Model:                            OLS   Adj. R-squared:                  0.083
Method:                 Least Squares   F-statistic:                     6.346
Date:                Wed, 01 Jul 2026   Prob (F-statistic):           2.73e-06
Time:                        00:16:40   Log-Likelihood:                -1013.6
No. Observations:                 297   AIC:                             2041.
Df Residuals:                     290   BIC:                             2067.
Df Model:                           6                                         
Covariance Type:                  HC3                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                     75

In [10]:
# I am checking multicollinearity between the Model 1 predictors.
# High VIF values would mean some predictors overlap too much and make coefficients unstable.

from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_1 = pd.DataFrame()
vif_1["Variable"] = X1.columns
vif_1["VIF"] = [variance_inflation_factor(X1.values, i) for i in range(X1.shape[1])]

vif_1

,Variable,VIF
0,const,570.707077
1,Population_Density,2.467068
2,Household_Size,1.644597
3,Literacy_Rate,1.423742
4,Total_Crime_per_100k,2.263640
5,Violent_Crime_per_100k,1.689617
6,Metro,2.390494


In [11]:
# I am creating a clean table from the Model 1 robust regression output.
# This is easier to read and save than the full statsmodels output.

def significance_stars(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    elif p < 0.10:
        return "."
    else:
        return ""

results_table_1 = pd.DataFrame({
    "Variable": model_1_robust.params.index,
    "Coefficient": model_1_robust.params.values,
    "Robust_Std_Error": model_1_robust.bse.values,
    "P_Value": model_1_robust.pvalues.values,
    "CI_Lower": model_1_robust.conf_int()[0].values,
    "CI_Upper": model_1_robust.conf_int()[1].values
})

results_table_1["Significance"] = results_table_1["P_Value"].apply(significance_stars)

results_table_1_clean = results_table_1.copy()
results_table_1_clean["Coefficient"] = results_table_1_clean["Coefficient"].round(4)
results_table_1_clean["Robust_Std_Error"] = results_table_1_clean["Robust_Std_Error"].round(4)
results_table_1_clean["P_Value"] = results_table_1_clean["P_Value"].round(4)
results_table_1_clean["CI_Lower"] = results_table_1_clean["CI_Lower"].round(4)
results_table_1_clean["CI_Upper"] = results_table_1_clean["CI_Upper"].round(4)

results_table_1_clean

,Variable,Coefficient,Robust_Std_Error,P_Value,CI_Lower,CI_Upper,Significance
0,const,75.2771,10.1987,0.0000,55.2881,95.2662,***
1,Population_Density,-0.0001,0.0001,0.0435,-0.0003,-0.0000,*
2,Household_Size,0.6930,1.3534,0.6086,-1.9595,3.3456,
3,Literacy_Rate,-0.0255,0.0844,0.7622,-0.1909,0.1398,
4,Total_Crime_per_100k,-0.1890,0.0481,0.0001,-0.2833,-0.0947,***
5,Violent_Crime_per_100k,0.1737,0.1779,0.3288,-0.1749,0.5223,
6,Metro,-1.4813,1.9325,0.4434,-5.2691,2.3064,


In [12]:
# I am saving the cleaned Model 1 table.
# This CSV can be uploaded to GitHub or used later in the written report.

results_table_1_clean.to_csv("OLS_Model_1_Robust_Results.csv", index=False)

os.listdir()

['Linear Regression.ipynb',
 'Main_Datasheet_Rem.xlsx',
 'OLS_Model_1_Robust_Results.csv',
 'Untitled Folder',
 'cpp-tiny-ray-tracer.ipynb',
 'cpp-third-party-libs.ipynb',
 'Lorenz.ipynb',
 'cpp.ipynb',
 'r.ipynb',
 'Intro.ipynb',
 'sqlite.ipynb']

In [13]:
# I am preparing Model 2 by adding the district-level remittance proxy.
# Remittance is not direct constituency-level data, so I will interpret it carefully later.

model_2_vars = [
    "Winner_Vote_Share",
    "Population_Density",
    "Household_Size",
    "Literacy_Rate",
    "Total_Crime_per_100k",
    "Violent_Crime_per_100k",
    "Metro",
    "Remittance_HH_Total"
]

reg_df_2 = df.loc[df["Use_For_Analysis"] == 1, model_2_vars].copy()

for col in model_2_vars:
    reg_df_2[col] = pd.to_numeric(reg_df_2[col], errors="coerce")

print(reg_df_2.shape)
reg_df_2.isna().sum()

(297, 8)


Winner_Vote_Share         0
Population_Density        0
Household_Size            0
Literacy_Rate             0
Total_Crime_per_100k      0
Violent_Crime_per_100k    0
Metro                     0
Remittance_HH_Total       0
dtype: int64

In [14]:
# Raw remittance household counts are very uneven across districts.
# I am using a log version to reduce skew and make the regression more stable.

reg_df_2["Log_Remittance_HH_Total"] = np.log1p(reg_df_2["Remittance_HH_Total"])

reg_df_2[["Remittance_HH_Total", "Log_Remittance_HH_Total"]].head()

,Remittance_HH_Total,Log_Remittance_HH_Total
0,13393,9.502562
1,13393,9.502562
2,13393,9.502562
3,13393,9.502562
4,4106,8.320448


In [15]:
# Model 2 adds the district-level remittance proxy to Model 1.
# The aim is to see whether remittance-heavy districts are associated with different winner vote share patterns.

y2 = reg_df_2["Winner_Vote_Share"]

X2 = reg_df_2[
    [
        "Population_Density",
        "Household_Size",
        "Literacy_Rate",
        "Total_Crime_per_100k",
        "Violent_Crime_per_100k",
        "Metro",
        "Log_Remittance_HH_Total"
    ]
]

X2 = sm.add_constant(X2)

model_2_robust = sm.OLS(y2, X2).fit(cov_type="HC3")

print(model_2_robust.summary())

                            OLS Regression Results                            
Dep. Variable:      Winner_Vote_Share   R-squared:                       0.105
Model:                            OLS   Adj. R-squared:                  0.084
Method:                 Least Squares   F-statistic:                     6.036
Date:                Wed, 01 Jul 2026   Prob (F-statistic):           1.39e-06
Time:                        00:24:22   Log-Likelihood:                -1012.9
No. Observations:                 297   AIC:                             2042.
Df Residuals:                     289   BIC:                             2071.
Df Model:                           7                                         
Covariance Type:                  HC3                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

In [17]:
# I am checking multicollinearity again after adding the remittance variable.
# This tells me whether remittance overlaps too strongly with the other predictors.

vif_2 = pd.DataFrame()
vif_2["Variable"] = X2.columns
vif_2["VIF"] = [variance_inflation_factor(X2.values, i) for i in range(X2.shape[1])]

vif_2

,Variable,VIF
0,const,741.930219
1,Population_Density,2.513403
2,Household_Size,1.647589
3,Literacy_Rate,1.523914
4,Total_Crime_per_100k,2.601121
5,Violent_Crime_per_100k,1.754398
6,Metro,2.390961
7,Log_Remittance_HH_Total,1.870715


In [18]:
# I am creating a clean table from the Model 2 robust regression output.
# This table is easier to read than the full statsmodels summary.

results_table_2 = pd.DataFrame({
    "Variable": model_2_robust.params.index,
    "Coefficient": model_2_robust.params.values,
    "Robust_Std_Error": model_2_robust.bse.values,
    "P_Value": model_2_robust.pvalues.values,
    "CI_Lower": model_2_robust.conf_int()[0].values,
    "CI_Upper": model_2_robust.conf_int()[1].values
})

results_table_2["Significance"] = results_table_2["P_Value"].apply(significance_stars)

results_table_2_clean = results_table_2.copy()
results_table_2_clean["Coefficient"] = results_table_2_clean["Coefficient"].round(4)
results_table_2_clean["Robust_Std_Error"] = results_table_2_clean["Robust_Std_Error"].round(4)
results_table_2_clean["P_Value"] = results_table_2_clean["P_Value"].round(4)
results_table_2_clean["CI_Lower"] = results_table_2_clean["CI_Lower"].round(4)
results_table_2_clean["CI_Upper"] = results_table_2_clean["CI_Upper"].round(4)

results_table_2_clean

,Variable,Coefficient,Robust_Std_Error,P_Value,CI_Lower,CI_Upper,Significance
0,const,68.6113,13.9463,0.0000,41.2771,95.9455,***
1,Population_Density,-0.0002,0.0001,0.0286,-0.0003,-0.0000,*
2,Household_Size,0.6214,1.3378,0.6423,-2.0005,3.2434,
3,Literacy_Rate,-0.0500,0.0845,0.5541,-0.2157,0.1157,
4,Total_Crime_per_100k,-0.1643,0.0589,0.0053,-0.2797,-0.0488,**
5,Violent_Crime_per_100k,0.2166,0.1836,0.2380,-0.1432,0.5765,
6,Metro,-1.4468,1.9227,0.4518,-5.2151,2.3216,
7,Log_Remittance_HH_Total,0.5471,0.6094,0.3693,-0.6473,1.7415,


In [19]:
# I am saving the cleaned Model 2 table.
# This file includes the regression with the district-level remittance proxy.

results_table_2_clean.to_csv("OLS_Model_2_With_Remittance_Robust_Results.csv", index=False)

os.listdir()

['Linear Regression.ipynb',
 'Main_Datasheet_Rem.xlsx',
 'OLS_Model_1_Robust_Results.csv',
 'OLS_Model_2_With_Remittance_Robust_Results.csv',
 'Untitled Folder',
 'cpp-tiny-ray-tracer.ipynb',
 'cpp-third-party-libs.ipynb',
 'Lorenz.ipynb',
 'cpp.ipynb',
 'r.ipynb',
 'Intro.ipynb',
 'sqlite.ipynb']